In [ ]:
import os
import cv2
import numpy as np
import math

color_dir = "colour"
laser_dir = "laser"
num_frames = 360   
output_xyz = "F11215109_second.xyz"

# first one is pilot
# src_pts = np.array([
#     [ 241,   86],
#     [ 486,  270],
#     [ 479,  970],
#     [ 368, 1041],
# ], dtype=np.float32)

src_pts = np.array([
    [ 240,   86],
    [ 486,  269],
    [ 261, 1109],
    [ 432, 1000],
], dtype=np.float32)


dst_pts = np.array([
    [ 25.0, 70.0 ],   # real‐world top‐left
    [ -25.0, 70.0 ],   # real‐world top‐right
    [ -25.0,  0.0 ],   # real‐world bottom‐right
    [ 25.0,  0.0 ]    # real‐world bottom‐left
], dtype=np.float32)

H, status = cv2.findHomography(src_pts, dst_pts)


all_points = []

for i in range(num_frames):
    filename = f"{i:04d}.jpg"
    color_path = os.path.join(color_dir, filename)
    laser_path = os.path.join(laser_dir, filename)

    # Read the “laser off” (background) image and the “laser on” image
    color_img = cv2.imread(color_path)    # background
    laser_img = cv2.imread(laser_path)    # with laser line
    if color_img is None or laser_img is None:
        print(f"[Warning] Could not read {filename}; skipping.")
        continue

    # 3.1 Subtract to isolate the green laser line
    diff = cv2.absdiff(laser_img, color_img)
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)

    # 3.2 Threshold and median‐blur to remove noise
    _, binary = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    binary = cv2.medianBlur(binary, 5)

    # 3.3 Extract all (row, col) of nonzero (laser) pixels
    laser_pixels = np.column_stack(np.where(binary > 0))  # shape = (N_nonzero, 2)
    if laser_pixels.shape[0] == 0:
        # If no laser detected in this frame, skip
        continue

    # 3.4 For each unique row, pick exactly one column via Gaussian weighting
    unique_rows = np.unique(laser_pixels[:, 0])  # rows that contain at least one laser pixel
    filtered_pixels = []
    for r in unique_rows:
        cols = laser_pixels[laser_pixels[:, 0] == r, 1]  # all columns in that row
        # Build a small 1D Gaussian (σ=3) around the mean column
        weights = np.exp(-0.5 * ((cols - np.mean(cols)) / 3.0) ** 2)
        weights /= np.sum(weights)
        x_gauss = np.round(np.sum(cols * weights)).astype(int)
        filtered_pixels.append([r, x_gauss])
    filtered_pixels = np.array(filtered_pixels)  # shape = (M, 2), each is [row, col_avg]

    # 3.5 Map each filtered_pixel from (col, row) → real‐world (X_plane_mm, Y_plane_mm) via Homography

    pts_img = filtered_pixels[:, [1, 0]].astype(np.float32).reshape(-1, 1, 2)
    pts_plane = cv2.perspectiveTransform(pts_img, H).reshape(-1, 2)

    # 3.6 Rotate each (X_plane, Y_plane, Z=0) around the world Y‐axis by θ = i degrees
    theta = np.deg2rad(i)  # i from 0..359 → θ in radians
    cos_t, sin_t = np.cos(theta), np.sin(theta)

    # Outer loop over each filtered point to compute its 3D world coords + color
    for idx, (X_plane, Y_plane) in enumerate(pts_plane):
        # 3D rotation around Y axis:
        X_w =  X_plane * cos_t       # X_world
        Y_w =  Y_plane               # Y_world stays the same
        Z_w =  X_plane * sin_t       # Z_world

        # 3.7 Sample the color from the background image at pixel (row, col)
        row_pix, col_pix = filtered_pixels[idx]  # (row, col)
        b, g, r = color_img[row_pix, col_pix]    # OpenCV uses BGR
        # Append (X, Y, Z, R, G, B)
        all_points.append([X_w, Y_w, Z_w, r, g, b])

print(f"Accumulated {len(all_points)} total 3D points.")


with open(output_xyz, "w") as f:
    for pt in all_points:
        X_w, Y_w, Z_w, r, g, b = pt
        # Write each line as: X Y Z R G B
        f.write(f"{X_w:.3f} {Y_w:.3f} {Z_w:.3f} {int(r)} {int(g)} {int(b)}\n")

print(f">>> Wrote point cloud to: {output_xyz}")


Accumulated 300142 total 3D points.
>>> Wrote point cloud to: F11215109.xyz
